01_Bronze_Incremental_Ingestion

In [0]:
from pyspark.sql.functions import current_timestamp, col
#recieve parameter from adf
dbutils.widgets.text("p_dataset_name", "")
dbutils.widgets.text("p_source_path", "")

dataset_name = dbutils.widgets.get("p_dataset_name")
source_path = dbutils.widgets.get("p_source_path")
# dbutild.widget.text("p_data_set")

#ADD PATH
storage_account = "statlasdev002"


# spark.conf.set(
#     f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
#     storage_account_key
# )
catalog_name = "dbw_atlas_dev_7405606293032023"

# ABFSS paths
full_source_path = f"abfss://landing@{storage_account}.dfs.core.windows.net/{dataset_name}/"
checkpoint_path = f"abfss://bronze@{storage_account}.dfs.core.windows.net/_checkpoints/{dataset_name}"

# stream read using autoloader(cloudFiles)
df_raw  = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", f"{checkpoint_path}/schema")
    .option("header", "true")
    .option("cloudFiles.inferColumnTypes", "true")
    .load(full_source_path)
)
# 4. Add Audit Columns
df_enriched = df_raw \
    .withColumn("_ingestion_timestamp", current_timestamp()) \
    .withColumn("_source_file_path", col("_metadata.file_path"))

# 5. Write Stream to Unity Catalog Bronze Table
external_table_path = f"abfss://bronze@{storage_account}.dfs.core.windows.net/{dataset_name}"
(df_enriched.writeStream
    .format("delta")
    .option("checkpointLocation", f"{checkpoint_path}/data")
    .option("mergeSchema", "true")
    .trigger(availableNow=True) 
    .option("path", external_table_path)
    .toTable(f"{catalog_name}.bronze.{dataset_name}") 
)






In [0]:
# %sql
# SELECT * FROM dbw_atlas_dev_7405606293032023.bronze.orders LIMIT 10;

In [0]:
# %sql
# select count(*)

In [0]:
# %sql
# drop table dbw_atlas_dev_7405606293032023.bronze.orders

In [0]:
# %sql
# drop volume dbw_atlas_dev_7405606293032023.bronze.raw_files;

In [0]:
# # Clear old data path
# dbutils.fs.rm("abfss://bronze@statlasdev002.dfs.core.windows.net/orders", True)

# # Clear old checkpoint path
# dbutils.fs.rm("abfss://bronze@statlasdev002.dfs.core.windows.net/_checkpoints/orders", True)

In [0]:
%sql
SELECT COUNT(*) FROM dbw_atlas_dev_7405606293032023.bronze.orders;

In [0]:
# # Yeh code dataframe save hone ke baad chalega
# # Maan lo tumhara final dataframe df hai jo save hua hai
# try:
#     rows_processed = df.count()
# except:
#     rows_processed = 0

# # Yeh line value ko wapas ADF ke paas bhej degi
# dbutils.notebook.exit(str(rows_processed))

In [0]:
# spark.sql("TRUNCATE TABLE dbw_atlas_dev_7405606293032023.bronze.shipments")

In [0]:
# checkpoint_path = f"abfss://bronze@statlasdev002.dfs.core.windows.net/_checkpoints/shipments"
# dbutils.fs.rm(checkpoint_path,True)

In [0]:
try:
    row_processed = df_enriched.count()
except:
    row_processed =0
dbutils.notebook.exit(str(row_processed))